In [2]:
import os
import json
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [2]:
train_src = Path("data/cut_yalur_green_v2")
valid_src = Path("data/cut_moot_100m")

output_root = Path("data/rf_detr_dataset")

train_out = output_root / "train"
valid_out = output_root / "valid"

train_out.mkdir(parents=True, exist_ok=True)
valid_out.mkdir(parents=True, exist_ok=True)

In [4]:
CATEGORIES = [
    {"id": 0, "name": "penguin"}
]

In [4]:
def yolo_to_coco(dataset_path, output_path, start_image_id=0, start_ann_id=0):
    
    images = []
    annotations = []
    
    image_id = start_image_id
    ann_id = start_ann_id
    
    images_dir = dataset_path / "images"
    labels_dir = dataset_path / "labels"
    
    for img_path in tqdm(list(images_dir.glob("*"))):
        
        new_name = f"{image_id:06d}{img_path.suffix}"
        dst_img = output_path / new_name
        shutil.copy(img_path, dst_img)
        
        w, h = Image.open(img_path).size
        
        images.append({
            "id": image_id,
            "file_name": new_name,
            "width": w,
            "height": h
        })
        
        label_path = labels_dir / (img_path.stem + ".txt")
        
        if label_path.exists():
            with open(label_path) as f:
                lines = f.readlines()
            
            for line in lines:
                cls, x, y, bw, bh = map(float, line.split())
                
                x_min = (x - bw/2) * w
                y_min = (y - bh/2) * h
                width = bw * w
                height = bh * h
                
                annotations.append({
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": int(cls),
                    "bbox": [x_min, y_min, width, height],
                    "area": width * height,
                    "iscrowd": 0
                })
                
                ann_id += 1
        
        image_id += 1
    
    return images, annotations, image_id, ann_id

In [5]:
train_images = []
train_annotations = []

img_id = 0
ann_id = 0

imgs, anns, img_id, ann_id = yolo_to_coco(
    train_src,
    train_out,
    img_id,
    ann_id
)

train_images += imgs
train_annotations += anns

100%|██████████| 2007/2007 [00:01<00:00, 1101.92it/s]


In [6]:
valid_images = []
valid_annotations = []

img_id = 0
ann_id = 0

imgs, anns, img_id, ann_id = yolo_to_coco(
    valid_src,
    valid_out,
    img_id,
    ann_id
)

valid_images += imgs
valid_annotations += anns

100%|██████████| 891/891 [00:00<00:00, 995.97it/s] 


In [7]:
train_coco = {
    "images": train_images,
    "annotations": train_annotations,
    "categories": CATEGORIES
}

with open(train_out / "_annotations.coco.json", "w") as f:
    json.dump(train_coco, f)

In [8]:
valid_coco = {
    "images": valid_images,
    "annotations": valid_annotations,
    "categories": CATEGORIES
}

with open(valid_out / "_annotations.coco.json", "w") as f:
    json.dump(valid_coco, f)

In [3]:
base_dataset = Path("data/rf_detr_dataset")

uk_dataset = Path("data/cut_uk_dataset")
unlabeled_dataset = Path("data/cut_unlabeled_dataset")

output_root = Path("data/rf_detr_uk_unlabeled")

train_out = output_root / "train"
valid_out = output_root / "valid"

train_out.mkdir(parents=True, exist_ok=True)
valid_out.mkdir(parents=True, exist_ok=True)

In [5]:
def load_coco(path):
    with open(path) as f:
        return json.load(f)

base_train = load_coco(base_dataset / "train/_annotations.coco.json")
base_valid = load_coco(base_dataset / "valid/_annotations.coco.json")

train_images = base_train["images"]
train_annotations = base_train["annotations"]

valid_images = base_valid["images"]
valid_annotations = base_valid["annotations"]

In [6]:
def copy_existing_images(src, dst):
    for img in src.glob("*.*"):
        shutil.copy(img, dst)

copy_existing_images(base_dataset / "train", train_out)
copy_existing_images(base_dataset / "valid", valid_out)

In [7]:
train_img_id = max(x["id"] for x in train_images) + 1
train_ann_id = max(x["id"] for x in train_annotations) + 1

valid_img_id = max(x["id"] for x in valid_images) + 1
valid_ann_id = max(x["id"] for x in valid_annotations) + 1

In [11]:
def append_yolo_split(dataset_path, split, output_path,
                      images, annotations, image_id, ann_id):

    images_dir = dataset_path / "images" / split
    labels_dir = dataset_path / "labels" / split

    img_files = list(images_dir.glob("*"))
    print(f"{split} images:", len(img_files))

    for img_path in tqdm(img_files):

        new_name = f"{image_id:06d}{img_path.suffix}"
        shutil.copy(img_path, output_path / new_name)

        w, h = Image.open(img_path).size

        images.append({
            "id": image_id,
            "file_name": new_name,
            "width": w,
            "height": h
        })

        label_path = labels_dir / (img_path.stem + ".txt")

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.split())

                    x_min = (x - bw/2) * w
                    y_min = (y - bh/2) * h
                    width = bw * w
                    height = bh * h

                    annotations.append({
                        "id": ann_id,
                        "image_id": image_id,
                        "category_id": int(cls),
                        "bbox": [x_min, y_min, width, height],
                        "area": width * height,
                        "iscrowd": 0
                    })

                    ann_id += 1

        image_id += 1

    return image_id, ann_id

In [13]:
train_img_id, train_ann_id = append_yolo_split(
    uk_dataset,
    "train",
    train_out,
    train_images,
    train_annotations,
    train_img_id,
    train_ann_id
)

valid_img_id, valid_ann_id = append_yolo_split(
    uk_dataset,
    "val",
    valid_out,
    valid_images,
    valid_annotations,
    valid_img_id,
    valid_ann_id
)

train images: 589


100%|██████████| 589/589 [00:00<00:00, 898.16it/s]


val images: 28


100%|██████████| 28/28 [00:00<00:00, 899.83it/s]


In [14]:
def append_unlabeled_to_train():

    global train_img_id, train_ann_id

    images_dir = unlabeled_dataset / "images"
    labels_dir = unlabeled_dataset / "labels"

    img_files = list(images_dir.glob("*"))
    print("unlabeled images:", len(img_files))

    for img_path in tqdm(img_files):

        new_name = f"{train_img_id:06d}{img_path.suffix}"
        shutil.copy(img_path, train_out / new_name)

        w, h = Image.open(img_path).size

        train_images.append({
            "id": train_img_id,
            "file_name": new_name,
            "width": w,
            "height": h
        })

        label_path = labels_dir / (img_path.stem + ".txt")

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.split())

                    x_min = (x - bw/2) * w
                    y_min = (y - bh/2) * h
                    width = bw * w
                    height = bh * h

                    train_annotations.append({
                        "id": train_ann_id,
                        "image_id": train_img_id,
                        "category_id": int(cls),
                        "bbox": [x_min, y_min, width, height],
                        "area": width * height,
                        "iscrowd": 0
                    })

                    train_ann_id += 1

        train_img_id += 1

In [15]:
append_unlabeled_to_train()

unlabeled images: 1910


100%|██████████| 1910/1910 [00:01<00:00, 1582.75it/s]


In [16]:
with open(train_out / "_annotations.coco.json", "w") as f:
    json.dump({
        "images": train_images,
        "annotations": train_annotations,
        "categories": CATEGORIES
    }, f)

with open(valid_out / "_annotations.coco.json", "w") as f:
    json.dump({
        "images": valid_images,
        "annotations": valid_annotations,
        "categories": CATEGORIES
    }, f)

In [17]:
dataset = Path("data/rf_detr_uk_unlabeled/train")
ann_path = dataset / "_annotations.coco.json"

with open(ann_path) as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

print("images:", len(images))
print("annotations:", len(annotations))

images: 4506
annotations: 43437


In [18]:
image_ids_with_labels = set(a["image_id"] for a in annotations)
all_image_ids = set(i["id"] for i in images)

no_label = all_image_ids - image_ids_with_labels

print("images without labels:", len(no_label))

images without labels: 2093
